<a href="https://colab.research.google.com/github/hthomas229/PurpleCrown/blob/main/polars_tutorial_1thru19.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#POLARS TUTORIAL

##Intro to Polars and Your First DataFrame

🐻‍❄️ Install and import polars


In [ ]:
!pip install polars

In [ ]:
import polars as pl
from rich import print #for pretty printing)

Create a polars series

In [ ]:
series = pl.Series("fruits", ["apple", "banana", "cherry"]) #column name and list of values
print(series)


Create a Polars Data Frame

In [ ]:
df = pl.DataFrame(  # a dictionary with column names as keys and values as lists
    {
        "name" :  ["Raj", "Bev", "Ken"],
        "age"  :  [11, 31, 42],
        "height": [4.5, 5.2, 6.3],
        "is_employed": [False, True, True]

    }
)
print(df)

🐻‍❄️ The first big difference between Polars and Pandas becomes obvious here:  NO INDEX.  This means no loc[] and no iloc[].

Polars provides a schema function for a quick glance


In [ ]:
print(df.schema)

Get a Glimpse of the Data

In [ ]:
df.glimpse()

Other common pandas functions like head and describe work also

In [ ]:
print(df.head(2))

In [ ]:
print(df.describe())

##Select Context Operations on DataFrames

🐻‍❄️ Note:  Polars is immutable. When we create result below, it returns a new dataframe.  There is no in place modification.

Select Specific Columns

In [ ]:
result = df.select("name", "age") # you can also pass a list of strings ["name", "age"]
print(result)

Rename Columns with Alias

In [ ]:
result = df.select(
    "name",
    pl.col("is_employed").alias("working?") # we use pl.col when we want to perform operations on a column

)
print(result)

Perform Basic Calculations

In [ ]:
#double the age
result = df.select(
    "name",
    pl.col("age") * 2
)
print(result)

Derived Column in Select

In [ ]:
result = df.select(
    "name",
    "height",
    (pl.col("height") > 6.0).alias("Tall")
)

print(result)

Create a Literal (Constant Column)

In [ ]:
result = df.select(
    pl.col("name"),
    pl.lit("name").alias("value")
)
print(result)

##Filtering Polars DataFrames


🐻‍❄️For the next few tutorials we will be using pokemon data. I will place the data file in my github repository https://github.com/hthomas229/PurpleCrown. Download it and upload to CoLab each time you run the notebook.

Scan Data from CSV - Lazy

🐻‍❄️ Lazy evaluation is one of the most powerful features of Polars.  We will cover it in detail in later tutorials.

In [ ]:
lf = pl.scan_csv("PokemonData.csv") #scans the csv contents
print(lf)

Read Data from CSV - Eager

🐻‍❄️ Since we are working with a small dataset we will load eager for the next few tutorials.



In [ ]:
df = pl.read_csv("PokemonData.csv") #reads the whole file into memory
print(df.head())

Basic Filter

In [ ]:
#Find fast pokemon
fast_pokemon = df.filter(pl.col("Speed") > 100).select("Name", "Speed")
print(fast_pokemon)

Multiple Conditions

`Polars uses ==, !=, &, | `

In [ ]:
result = df.filter(
    (pl.col("Attack") > 90) & (pl.col("Defense") > 80),

)
print(result)

String Filtering

In [ ]:
result = df.filter(
    pl.col("Type1") == "Fire"
    )

print(result)

Many Other String Methods



In [ ]:
result = df.filter(
    pl.col("Name").str.contains("saur")
)
print(result)

There will be a whole tutorial on string methods later.

Boolean Filtering

In [ ]:
#find pokemon who are Legendary or have a Speed higher than 130
result = df.filter(pl.col("Legendary") | (pl.col("Speed") >= 130)) # here "Legendary" only returns True values
print(result)

## **with_columns:** Adding Custom Columns to Polars DataFrames

🐻‍❄️with_columns creates a new dataframe that contains the columns from the original dataframe and the new columns according to its input expressions

In [ ]:
print(df.schema)

Add a custom column

In [ ]:
# add a column TotAtk which equals Attack + SpAtk
result = df.with_columns(
    (pl.col("Attack") + pl.col("SpAtk")).alias("TotAtk")
)
print(result)

In [ ]:
#select only Name and TotAtk -- chaining context -- we start to see pipelines
result = df.with_columns(
    (pl.col("Attack") + pl.col("SpAtk")).alias("TotAtk")
).select("Name", "TotAtk")
print(result)


Using a Variable & Adding Multiple Columns

In [ ]:
total_power = (pl.col("Attack") + pl.col("Speed")  + pl.col("Defense"))

result = df.with_columns(
    total_power = total_power,
    avg_total_power = total_power.mean(), #round for example in video
    max_total_power = total_power.max()
)

print(result)

Create a Conditioal Column with pl.when().then().otherwise()  -- Case statement

In [ ]:
#create a SpeedTier column

result = df.with_columns(
    pl.when(pl.col("Speed") >= 100).then(pl.lit("Fast"))
    .when(pl.col("Speed") >= 70).then(pl.lit("Medium Speed"))
    .otherwise(pl.lit("Slow")).alias("SpeedTier")
  )
print(result)

##Sorting and Limiting Polars DataFrames

In [ ]:
#sort pokemon by name
result =df.sort("Name") # ascending is the default argument for sort
print(result.head())

In [ ]:
#sort pokemon by Speed descending and return the top 5

result = df.sort("Speed",descending=True).limit(5)
print(result)

In [ ]:
#sort by HP descending and return the five weakest pokemon -- select only name and HP

result = df.sort("HP", descending=True).select("Name", "HP").tail(5)
print(result)

Multi Column Sort

In [ ]:
# sort by Type 1 ascending then Attack Descending -- select only Name, Type1 and Attack

result = df.sort(
    ["Type1", "Attack"],
    descending=[False, True]
).select("Name", "Type1", "Attack")

print(result)

Top and Bottom K

In [ ]:
#get 5 fastest pokemon
result = df.top_k(5, by="Speed")
print(result)

In [ ]:
#get 5 slowest pokemon
result = df.bottom_k(5, by="Speed")
print(result)

##Grouping & Aggregating Polars DataFrames

Here are some of the most commonly used aggregation functions: `sum(),mean(),median(), min(), max(), len()`



In [ ]:
# sum pokemon HP by Type1 and rename to TotHP
result = df.group_by("Type1").agg(
    pl.col("HP").sum().alias("TotHP")
)
print(result)

In [ ]:
# group by Type1 and get the Average Attack,Average Speed, Median Defense, Max HP and Min HP

result= (df.group_by("Type1")
.agg(
    pl.col("Attack").mean().alias("AvgAttack"),
    pl.col("Speed").mean().alias("AvgSpeed"),
    pl.col("Defense").median().alias("MedianDefense"),
    pl.col("HP").max().alias("MaxHP"),
    pl.col("HP").min().alias("MinHP"))
)

print(result)

In [ ]:
# get the count of each Type1 pokemon
result = df.group_by("Type1").agg(
    pl.len().alias("Count")
)

print(result)

In [ ]:
#group by speed in 10s and get max and min for each
result = df.group_by(
    (pl.col("HP") // 10 * 10).alias("SpeedTier")
).agg(
    pl.len(),
    pl.col("HP").max().alias("fastest"),
    pl.col("HP").min().alias("slowest")
).sort("SpeedTier", descending=True)
print(result)

#Lazy Evaluation

##The Real Power of Polars

We can scan and manipulate huge amounts of data that Pandas can't.

📊 Data Scale: pandas vs Polars
🐼 How much data can pandas handle?
Rule of thumb
pandas is memory-bound
If it doesn’t fit in RAM → ⚠️ you’re in trouble
Limits in practice
✅ Comfortable: hundreds of MB
⚖️ OK with tuning: 1–5 GB
😬 Painful / risky: 10+ GB
🛑 Hard stop: data size ≈ available RAM
Why pandas struggles at scale
Eager execution (everything loads immediately)
Python objects + NumPy arrays → high memory overhead
No native query optimizer
Frequent data copies (often invisible)
You can stretch pandas with:
Chunking (chunksize)
Careful dtype optimization
categorical columns
…but at the cost of manual complexity and fragile pipelines.

⚡ How much data can Polars handle?
Rule of thumb
Polars scales beyond RAM
Limits in practice
✅ Comfortable: many GB
🚀 Common: 10s–100s of GB
⏳ Realistic upper bound: disk space + patience
Why Polars scales better
Written in Rust
Uses Apache Arrow (columnar, cache-friendly)
Lazy execution + query optimizer
Streaming execution
Zero-copy operations where possible
Multi-threaded by default

In [ ]:
import polars as pl
from rich import print


# Ways to Create a LazyFrame


1.   From dictionary
2.   From DataFrame with .lazy()
3.   pl.scan_



In [ ]:
lf = pl.LazyFrame(
    {
        "tool" : ["hammer", "saw", "wrench"],
        "qty": [3, 5, 4]

    }
)
print(lf)

In [ ]:
lf = pl.DataFrame(
    {        "tool" : ["hammer", "saw", "wrench"],
        "qty": [3, 5, 4]

    }
).lazy()
print(lf)

In [ ]:
lf = pl.scan_csv("orders.csv")
print(lf)

To display the result we use the collect() method

In [ ]:
lf = pl.LazyFrame(
    {
        "tool" : ["hammer", "saw", "wrench"],
        "qty": [3, 5, 4]

    }
)
print(lf)

print(lf.collect())



| Optimization          | Explanation                                                                 | Runs  |
|-----------------------|-----------------------------------------------------------------------------|-------|
| Predicate Pushdown    | Applies filters as early as possible at the scan level.                     | 1x    |
| Projection Pushdown   | Selects only the required columns at the scan level.                        | 1x    |
| Slice Pushdown        | Loads only the required slice at scan time and avoids materializing slices  | 1x    |

### Why this matters
- **Less data read** → faster execution  
- **Lower memory usage** → better scalability  
- **Works best with lazy execution engines** (e.g., Polars, Spark)


Simple Pushdown Example

In [ ]:
query = lf.filter(
    pl.col("tool") == "hammer"
)
print(query)

In [ ]:
print(query.collect())

In [ ]:
query = lf.group_by("tool").agg("qty").sum()
print(query)

In [ ]:
query.explain(optimized=True)

Graph

In [ ]:
query = lf.group_by("tool").agg("qty").sum()
print(query.show_graph())

Execution

In [ ]:
lf = pl.scan_csv("orders.csv")
print(lf)

In [ ]:
print(lf.collect().limit(5))

In [ ]:
q = (
    pl.scan_csv("orders.csv")
   .with_columns(pl.col("product").str.to_lowercase())
  .filter(pl.col("product")== "laptop")
     )
print(q.explain())

In [ ]:
print(q.show_graph())

In [ ]:
print(q.collect())

If we we have more data than will fit in RAM we can pass the engine = "streaming"  argument to collect

In [ ]:
q = (
    pl.scan_csv("orders.csv")
   .with_columns(pl.col("product").str.to_lowercase())
  .filter(pl.col("product")== "laptop")
     ).collect(engine="streaming")
print(q)

##Basic Data Types & Casting






In [ ]:
import polars as pl
from rich import print

Integers (Int64, UInt32)

Floats (Float64)

Strings (Utf8)

Boolean

Date / Datetime / Duration

Categorical

Pyhton implicilty infers data types.  You can declare them implicitly using a schema/

In [ ]:
import polars as pl
from datetime import date, datetime, timedelta
pl.Config.set_tbl_cols(20)

schema = {
    "int_signed": pl.Int64,
    "int_unsigned": pl.UInt32,
    "float_col": pl.Float64,
    "string_col": pl.Utf8,
    "bool_col": pl.Boolean,
    "date_col": pl.Date,
    "datetime_col": pl.Datetime,
    "duration_col": pl.Duration,
    "category_col": pl.Categorical,
}

In [ ]:
from datetime import date, datetime, timedelta

df = pl.DataFrame(
    {
        "int_signed": [1, -5, 10],
        "int_unsigned": [1, 5, 10],
        "float_col": [1.5, 2.0, 3.75],
        "string_col": ["apple", "banana", "cherry"],
        "bool_col": [True, False, True],
        "date_col": [
            date(2024, 1, 1),
            date(2024, 1, 2),
            date(2024, 1, 3),
        ],
        "datetime_col": [
            datetime(2024, 1, 1, 9, 0),
            datetime(2024, 1, 2, 10, 30),
            datetime(2024, 1, 3, 14, 45),
        ],
        "duration_col": [
            timedelta(days=1),
            timedelta(hours=2),
            timedelta(minutes=30),
        ],
        "category_col": ["A", "B", "A"],
    },
    schema=schema,
)

print(df)
print(df.dtypes)


Casting


In [ ]:
df = pl.DataFrame({ "a": ["1", "2", "3"], "b": ["2026-01-01", "2026-01-02", "2026-01-03"] })
print(df)

In [ ]:
result = df.with_columns([ pl.col("a").cast(pl.Int64), pl.col("b").cast(pl.Date) ])
print(result)

SAFE v. STRICT

Polars can either:

strict: fail on invalid values

non‑strict: turn invalid values into nulls

In [ ]:
df = pl.DataFrame(
    {
        "raw_numbers": ["10", "20", "oops", "30"]
    }
)

print(df)
print(df.dtypes)

In [ ]:
#default strict=true
result = df.with_columns(
    pl.col("raw_numbers").cast(pl.Int64)
)

In [ ]:
result = df.with_columns(
    pl.col("raw_numbers").cast(pl.Int64, strict=False)
)
print(result)

##String Methods & Parsing

In [ ]:
import polars as pl
from rich import print

df = pl.DataFrame(
    {
        "tools": ["hammer🔨", "saw", "wrench", "screwdriver"],

    }
)

print(df)

len_bytes & len_chars

In [ ]:
result = df.with_columns(
    pl.col("tools").str.len_bytes().alias("byte_count"),
    pl.col("tools").str.len_chars().alias("letter_count"),

)
print(result)

Pattern Matching

In [ ]:
result = df.select(
    pl.col("tools"),
    pl.col("tools").str.starts_with("h").alias("starts_wtih_h"),
    pl.col("tools").str.contains("s..r").alias("s..r"),
    pl.col("tools").str.contains("e+").alias("e+"),
    pl.col("tools").str.ends_with("r").alias("ends_with_r"),


)
print(result)

String Methods




In [ ]:
words = pl.DataFrame(
    {
        "word": ["GEorge", "LIKES", "his", "cHickEn", "SpicY!"]
    }
)
print(words)

In [ ]:
result = words.with_columns(
    pl.col("word").str.to_lowercase().alias("lower"),
    pl.col("word").str.to_uppercase().alias("upper"),
    pl.col("word").str.to_titlecase().alias("title"),
    pl.col("word").str.zfill(10).alias("z_fill"),

)
print(result)

Explode

In [ ]:
test = pl.DataFrame(
    {
    "A": ["Spicy", "Chicken", "Antidentite"]
}
)
print(test.select(pl.col("A").str.explode()))

Padded

In [ ]:
result = test.with_columns(
    pl.col("A").str.pad_start(11, "*").alias("padded")
)
print(result)

Replace


In [ ]:
result = test.with_columns(
    pl.col("A").str.replace("ken", "ago").alias("replaced")
)
print(result)

Reverse

In [ ]:
result = test.with_columns(
    pl.col("A").str.reverse().alias("backwards")
)
print(result)

Slice

In [ ]:
print(test.select("A", substr=pl.col("A").str.slice(3, length=2)))

strip_prefix and strip_suffix

In [ ]:
print(test.select(pl.col("A").str.strip_prefix("Chi").alias("stripped")))

In [ ]:
print(test.select(pl.col("A").str.strip_suffix("ite").alias("stripped")))

Extract


In [ ]:
product_codes = pl.DataFrame(
    {
        "item_description": [
            "Product_A_ID:123",
            "Product_B_ID:456_v2",
            "Product_C_NoID",
            "Product_D_ID:789-X",
            "Product_E"
        ]
    }
)
print(product_codes)

In [ ]:
# Extract the numerical ID following 'ID:' using a regex pattern
# The regex r'ID:(\d+)' captures one or more digits after 'ID:'
result = product_codes.with_columns(
    pl.col("item_description").str.extract(r"ID:(\d+)").alias("extracted_id")
)
print(result)

Parsing Date and Time Strings

In [ ]:
clock = pl.DataFrame(
    {
        "time": ["10:00", "11:00", "12:00"]
    }
)
print(clock)

In [ ]:
print(clock.with_columns(pl.col("time").str.strptime(pl.Time,'%H:%M').alias("formatted")))

In [ ]:
calendar = pl.DataFrame(
    {
        "date": ["2026-05-10", "2026-07-04", "2026-10-31"]
    }
)
print(calendar)

In [ ]:
result = calendar.with_columns(
    pl.col("date").str.strptime(pl.Date, "%Y-%m-%d").alias("formatted")
)
print(result)

##Dates and Times


In [ ]:
from datetime import datetime, date

df = pl.DataFrame({
    "datetime" : [datetime(2025, 1, 1,3,30,56), datetime(2024,12,25, 5, 28, 47)]
})

df = df.with_columns([
    pl.col("datetime").dt.year().alias("year"),
    pl.col("datetime").dt.month().alias("month"),
    pl.col("datetime").dt.day().alias("day"),
    pl.col("datetime").dt.hour().alias("hour"),
    pl.col("datetime").dt.minute().alias("minute"),
    pl.col("datetime").dt.second().alias("second")
])
df

In [ ]:
from datetime import datetime
import polars as pl

df = pl.DataFrame({
     "start" : [datetime(2024,12,25, 5, 28, 47)],
     "end":   [datetime(2025, 1, 1,3,30,56)],
 })

df = df.with_columns(
    (pl.col("end") - pl.col("start")).alias("duration")
)
df

In [ ]:
df = pl.DataFrame(
    {
        "date": [date(2022, 1, 1), date(2022, 1, 2)],
        "string": ["2022-01-01", "2022-01-02"],
    }
)

result = df.select(
    pl.col("date").dt.to_string("%Y-%m-%d"),
    pl.col("string").str.to_datetime("%Y-%m-%d"),
)
print(result)

In [ ]:
result = df.select(
    pl.col("date").cast(pl.Int64).alias("days_since_epoch"),
    pl.col("datetime").cast(pl.Int64).alias("us_since_epoch"),
    pl.col("time").cast(pl.Int64).alias("ns_since_midnight"),

)
print(result)

In [ ]:
df = pl.DataFrame(
    {
        "date": [date(2022, 1, 1), date(2022, 1, 2)],
        "string": ["2022-01-01", "2022-01-02"],
    }
)

result = df.select(
    pl.col("date").dt.to_string("%Y-%m-%d"),
    pl.col("string").str.to_datetime("%Y-%m-%d"),
)
print(result)

Use Duration

In [ ]:
times = pl.DataFrame(
    {
        "driver":["Senna", "Prost", "Lauda", "Senna", "Prost", "Lauda"],
        "lap_time": ["1:23", "1:22", "1:21", "1:20", "1:21", "1:22"]
    }
)

times = times.with_columns(
    pl.col("lap_time").str.split(":").list.get(0).cast(pl.Int64).alias("minutes"),
    pl.col("lap_time").str.split(":").list.get(1).cast(pl.Int64).alias("seconds")
)

print(times)

In [ ]:
# Convert minutes and seconds to a Polars Duration object per lap
times_with_duration = times.with_columns(
    (pl.duration(minutes=pl.col("minutes"), seconds=pl.col("seconds"))).alias("lap_duration")
)

# Group by driver and sum the lap_duration
total_duration_by_driver = times_with_duration.group_by("driver").agg(
    pl.col("lap_duration").sum().alias("total_duration")
)

print(total_duration_by_driver)

##Null Handling

## 📦 Topics Covered

- Counting NULL values per column and row  
- Filtering rows with NULLs  
- Dropping NULLs (`drop_nulls`)  
- Filling NULLs with constants  
- Forward and backward filling  
- Conditional filling with expressions  
- Interpolating missing values  
- NULL vs NaN differences in Polars  
- LazyFrame optimizations for NULL handling  

## ⚡ Why Polars for Missing Data?

- Expression-based NULL handling  
- Works seamlessly in LazyFrames  
- Predicate and projection pushdown  
- Faster and more memory-efficient than pandas  


In [ ]:
import polars as pl
import numpy as np

df = pl.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "sales": [100.0, None, 300.0, None, 500.0],
    "region": ["US", "EU", None, "US", None],
    "growth": [0.10, np.nan, 0.30, None, 0.50],
})

print(df)


Counting Nulls

In [ ]:
#count null values per column
result = df.select(pl.all().null_count())
print(result)

In [ ]:
#with explicit labels
result = df.select(
    pl.all().null_count().name.suffix("_nulls")
)
print(result)

In [ ]:
#per row
result = df.with_columns(
    pl.sum_horizontal(pl.all().is_null()).alias("nulls_per_row")
)

print(result)

Filtering Nulls


In [ ]:
#by row
result = df.filter(
    pl.any_horizontal(pl.all().is_null())
)
print(result)

In [ ]:
#by row not null
result = df.filter(
    pl.all_horizontal(pl.all().is_not_null())
)
print(result)

In [ ]:
#specific column
result = df.filter(
    pl.col("region").is_null()
)
print(result)

Dropping Nulls -- Be Careful! ⚠️

In [ ]:
#drop any row with any nulls
result = df.drop_nulls()
print(result)

In [ ]:
#drop any row with NaNs
result = df.drop_nans()
print(result)

In [ ]:
#drop only row with nulls in sales and growth
result = df.drop_nulls(subset=["sales", "growth"])
print(result)

Filling Nulls

In [ ]:
#with a constant
result = df.with_columns([
    pl.col("sales").fill_null(0),
    pl.col("region").fill_null("Unknown")
])

print(result)

In [ ]:
#forward
result = df.with_columns(
    pl.col("sales").fill_null(strategy="forward")
)
print(result)

In [ ]:
#backward
result = df.with_columns(
    pl.col("sales").fill_null(strategy="backward")
)
print(result)

In [ ]:
#conditional
result = df.with_columns(
    pl.when(pl.col("sales").is_null())
    .then(pl.col("sales").mean())
    .otherwise(pl.col("sales"))
    .alias("sales_filed_mean")
)
print(result)

In [ ]:
#linear interpolation
result = df.with_columns(
    pl.col("sales").interpolate()
)
print(result)

More on NaNs

In [ ]:
#count NaNs
result = df.select(
    pl.col("growth").is_nan().sum().alias("nans")
)
print(result)

⭐ It is very common to first convert all NaNs to nulls and then clean them with the nulls.

In [ ]:
result = df.with_columns(
    pl.col("growth").fill_nan(None)
)
print(result)

##Expression Expansion

## 📝 Expression Expansion in Polars

**Expression Expansion** enables you to write a single expression that can expand into **multiple different expressions**, possibly depending on the **schema of the context** in which the expression is used.  

This feature isn’t just decorative or syntactic sugar — it’s a **powerful tool for DRY (Don’t Repeat Yourself) principles** in your code:

- A **single expression** can specify **multiple columns**.
- That expression **expands into a list of expressions**, meaning the computation is written **once** and **reused** everywhere it’s needed.
- Reduces boilerplate and keeps your Polars code **clean, maintainable, and efficient**.

> Example use case: compute a transformation across multiple numeric columns with **one expression**, without repeating the logic for each column.


In [ ]:
import polars as pl
from rich import print
pl.Config.set_tbl_rows(50)

In [ ]:
df = pl.DataFrame( {
    "day" : ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday" ],
    "am_high" : [32, 34, 33, 31, 29],
    "am_low"  : [23, 25, 26, 22, 25],
    "pm_high"  : [37, 36, 40, 38, 35],
    "pm_low"  : [24, 23, 25, 20, 26],
    "wind_speed" : [11.1, 12.3 ,9.5, 15.2, 8.0]
})
print(df)

Explicit expansion by column name

In [ ]:
result = df.with_columns(
    ((pl.col("am_high",
            "am_low",
            "pm_high",
            "pm_low") - 32) * (5/9)).round(1))

print(result)

Expansion by data type

In [ ]:
result = df.with_columns(
   ((pl.col(pl.Int64)- 32) * (5/9)).round(1)
   )
print(result)

All & Exclude

In [ ]:
result = df.select(pl.all())
print(result)

In [ ]:
result = df.select(pl.all().exclude("^am_.*$"))
print(result)

Renaming the New Columns

In [ ]:
result_celsius = df.with_columns(
    ((pl.col("^(am)_(high|low)$") - 32) * (5/9)).name.suffix("_celsius")
)
print(result_celsius)

Selectors

Polars comes with the submodule selectors that provides a number of functions that allow you to write more flexible column selections for expression expansion.


In [ ]:
import polars.selectors as cs

In [ ]:
result = df.select(cs.string() | cs.ends_with("_speed"))
print(result)

Combine Selectors with Expression Expansion


In [ ]:
result = df.select("day", (cs.contains("speed") - cs.string())* 2)
print(result)

##Unique and Value Counts in Polars DataFrames



In [ ]:
df = pl.DataFrame({
    "color": ["red", "blue", "red", "green", "blue", "blue"]
})


In [ ]:
print(df)

In [ ]:
# n_unique counts distinct values
print(df.select(pl.col("color")).n_unique())

In [ ]:
# unique returns the values themselves
print(df.select(pl.col("color").unique()))

In [ ]:
# value_counts returns one row per value with counts

print(df.select(pl.col("color").value_counts()))

#notice this returns a struct -- a data type we will discuss later

In [ ]:
#approx_n_unique tries to appproximate in cases of very large dataframes
import numpy as np

large_df = pl.DataFrame({"nums":  np.random.randint(0, 100_000, 100_000)})
result = large_df.select(
    pl.col("nums").n_unique().alias("n_unique"),
    pl.col("nums").approx_n_unique().alias("approx")
)

print(result)

##Window Functions


In [ ]:
!pip install -U polars

In [ ]:
import polars as pl
from rich import print
pl.Config.set_tbl_cols(50)
pl.Config.set_tbl_rows(50)

🐻‍❄️ Polars impicilty selects the data types when you create a DataFrame.
If you want to set them explicitly from the start declare a schema and use it as the 2nd parameter in the pl.DataFrame function.

In [ ]:

schema = {

        "name": pl.String,
        "team": pl.Categorical,
        "score": pl.Int32,
        "rating": pl.Float64


}

In [ ]:
df = pl.DataFrame({
    "name": ["Dyno Mutt", "Speed Buggy", "Captain Caveman", "Hong Kong Phooey", "Shaggy", "Yogi Bear", "Huckleberry Hound", "Grape Ape", "Doggy Daddy", "Quickdraw McGraw", "Mumbly", "Dirty Dalton", "Dinky Dalton", "Dastardly Dalton", "Mr. Creepley"],
    "team": ["Scoobie Doobies",  "Scoobie Doobies", "Scoobie Doobies", "Scoobie Doobies", "Scoobie Doobies", "Yogi Yahooies", "Yogi Yahooies", "Yogi Yahooies", "Yogi Yahooies", "Yogi Yahooies","Really Rottens", "Really Rottens", "Really Rottens", "Really Rottens", "Really Rottens"],
    "score": [31, 42, 34, 23, 42, 27, 36, 41, 42, 36, 11, 17, 11, 19, 21,],
    "rating" : [8.1, 9.5, 9.6, 9.9, 8.7, 8.1, 9.6, 9.9, 8.4, 9.5, 2.3, 4.0, 4.2, 3.1, 4.2  ]

}, schema=schema)

In [ ]:
print(df)

##Ranking

In [ ]:
ranked_df = df.with_columns(
    pl.col("score").rank().alias("score_rank")  #sort defaults to ascending, method defaults to average
).sort("score_rank")
print(ranked_df)

Method and Descending

In [ ]:
ranked_df = df.with_columns(
    pl.col("score").rank(descending=True, method="random" ).alias("score_rank")   #min, max, average, dense, ordian, random
).sort("score_rank")
print(ranked_df)

Over

In [ ]:
team_rank_df = df.with_columns(
    pl.col("score").rank(descending=True, method="ordinal" ).over("team").alias("score_rank")   #min, max, average, dense, ordian, random
).sort(["team", "score_rank"], descending=[True, False])
print(team_rank_df)

Aggregation Methods

In [ ]:
stats = df. with_columns([
    pl.col("rating").mean().over("team").alias("avg_score"),
    pl.col("rating").min().over("team").alias("min_score"),
    pl.col("rating").max().over("team").alias("max_score"),
    pl.col("rating").median().over("team").alias("median_score"),

])

print(stats)  #use to compare individual v group;

Running Total

In [ ]:
running_score = df.with_columns(
    pl.col("score").cum_count().over("team").alias("running_total")
).select("name", "team", "score", "running_total")

print(running_score)   #cum methods = sum, min, max, prod, count

Rolling Total

In [ ]:
rolling_3 = df.with_columns(
    pl.col("rating").rolling_max(window_size=3).alias("rolling_sum_rating")
)
print(rolling_3)   #sum, mean, mim, max, std, var, median, quartile

##Structs and Unnesting



The data type **Struct** is a composite data type that can store multiple fields in a single column.

When building series or dataframes, Polars will convert dictionaries to the data type **Struct**.

In [ ]:
df = pl.DataFrame(
    {
        "supe" : ["Spiderman", "Captain America", "Superman"],
        "details" : [ {"name": "Peter Parker", "age": 19, "power": "webslinging", "nemesis" : "Green Goblin"},
                    {"name": "Capt. Steve Rogers", "age": 31, "power": "shield", "nemesis": "Winter Soldier"},
                     {"name": "Clark Kent", "age": 42, "power": "superstrength", "nemesis": "Lex Luthor"}]
    }
)
print(df)

Access by Field

In [ ]:
result = df.select([
    pl.col("supe"),
    pl.col("details").struct.field("name").alias("name"),
    pl.col("details").struct.field("age").alias("age"),
    pl.col("details").struct.field("power").alias("power"),
    pl.col("details").struct.field("nemesis").alias("nemesis"),
])
print(result)

Unnest  🪹


In [ ]:
result = df.unnest(pl.col("details"))
print(result)

##Lists

In [ ]:
df = pl.DataFrame(
    {
    "names" : [["Paul", "John", "George", "Ringo"], ["Larry", "Moe", "Curly"], ["Sabrina", "Kelly", "Jill"]],
    "ages" : [[31, 32, 33, 34], [41, 42, 43], [27,  28, 39]]
    }
)

In [ ]:
print(df)

In [ ]:
result = df.explode(["names", "ages"])
print(result)

#The `Enum` Data Type




Enum is a fixed, ordered set of string categories known at creation time. Think of it like a Rust/C enum. They are stored efficiently as u32 indices internally

In [ ]:
import polars as pl
from rich import print

The enum must be defined and then declared as a data type on a pl.Series.

In [ ]:
languages_enum = pl.Enum(["Python", "Rust", "Javascript", "Go"])  #create the type

In [ ]:
languages = pl.Series(["Python", "Rust", "Go", "Rust", "Javascript", "Python"], dtype= languages_enum)
print(languages)

Error Handling

Python will raise an exception if the pl.col or Series includes a value not declared in the enum.

In [ ]:
languages = pl.Series(["Python", "Rust", "Stop", "Rust", "Javascript", "Python"], dtype= languages_enum)
print(languages)

In [ ]:
from polars.exceptions import InvalidOperationError

try:

  languages = pl.Series(["Python", "Rust", "Stop", "Rust", "Javascript", "Python"], dtype= languages_enum)
  print(languages)

except InvalidOperationError as e:
  print("InvalidOperationError ", e)

Comparison and Ordering

In [ ]:
level_enum = (["Low", "Medium", "High"])

In [ ]:
events = pl.DataFrame(
    {
        "Name":  ["Event1", "Event2", "Event3", "Event4"],
        "Level":  ["High", "Low", "Medium", "High"]
    },
    schema_overrides={"Level": pl.Enum(level_enum)}
)
print(events)

Enum columns are sorted in the order they were defined

In [ ]:
result = events.sort("Level")
print(result)


You can use comparison operators to filter etc.

In [ ]:
high_events = events.filter(pl.col("Level") >= "Medium")
print(high_events)

##Categorical Data

Categorical is for dynamic, open-ended string categories where you don't know all values upfront.


You can cast

In [ ]:
import polars as pl

df = pl.DataFrame({
    "task": ["crush", "kill", "destroy", "save"]
})

df = df.with_columns(
    pl.col("task").cast(pl.Categorical)
)

print(df)

shape: (4, 1)
┌─────────┐
│ task    │
│ ---     │
│ cat     │
╞═════════╡
│ crush   │
│ kill    │
│ destroy │
│ save    │
└─────────┘


Or declare in a schema like we did with Laff-a-Lympics

In [ ]:
schema = {

        "name": pl.String,
        "team": pl.Categorical,
        "score": pl.Int32,
        "rating": pl.Float64


}

In [ ]:
df = pl.DataFrame({
    "name": ["Dyno Mutt", "Speed Buggy", "Captain Caveman", "Hong Kong Phooey", "Shaggy", "Yogi Bear", "Huckleberry Hound", "Grape Ape", "Doggy Daddy", "Quickdraw McGraw", "Mumbly", "Dirty Dalton", "Dinky Dalton", "Dastardly Dalton", "Mr. Creepley"],
    "team": ["Scoobie Doobies",  "Scoobie Doobies", "Scoobie Doobies", "Scoobie Doobies", "Scoobie Doobies", "Yogi Yahooies", "Yogi Yahooies", "Yogi Yahooies", "Yogi Yahooies", "Yogi Yahooies","Really Rottens", "Really Rottens", "Really Rottens", "Really Rottens", "Really Rottens"],
    "score": [31, 42, 34, 23, 42, 27, 36, 41, 42, 36, 11, 17, 11, 19, 21,],
    "rating" : [8.1, 9.5, 9.6, 9.9, 8.7, 8.1, 9.6, 9.9, 8.4, 9.5, 2.3, 4.0, 4.2, 3.1, 4.2  ]

}, schema=schema)
print(df)

shape: (15, 4)
┌──────────────────┬─────────────────┬───────┬────────┐
│ name             ┆ team            ┆ score ┆ rating │
│ ---              ┆ ---             ┆ ---   ┆ ---    │
│ str              ┆ cat             ┆ i32   ┆ f64    │
╞══════════════════╪═════════════════╪═══════╪════════╡
│ Dyno Mutt        ┆ Scoobie Doobies ┆ 31    ┆ 8.1    │
│ Speed Buggy      ┆ Scoobie Doobies ┆ 42    ┆ 9.5    │
│ Captain Caveman  ┆ Scoobie Doobies ┆ 34    ┆ 9.6    │
│ Hong Kong Phooey ┆ Scoobie Doobies ┆ 23    ┆ 9.9    │
│ Shaggy           ┆ Scoobie Doobies ┆ 42    ┆ 8.7    │
│ …                ┆ …               ┆ …     ┆ …      │
│ Mumbly           ┆ Really Rottens  ┆ 11    ┆ 2.3    │
│ Dirty Dalton     ┆ Really Rottens  ┆ 17    ┆ 4.0    │
│ Dinky Dalton     ┆ Really Rottens  ┆ 11    ┆ 4.2    │
│ Dastardly Dalton ┆ Really Rottens  ┆ 19    ┆ 3.1    │
│ Mr. Creepley     ┆ Really Rottens  ┆ 21    ┆ 4.2    │
└──────────────────┴─────────────────┴───────┴────────┘


## Why use `Categorical`?

String columns store the full text of every value, for every row. `Categorical` instead stores each unique string **once**, with rows holding a small integer index — typically **5–10× less memory** when a column has few unique values relative to its length.

Filters, groupbys, and sorts also run faster since they operate on integers under the hood.

In [ ]:
import time
import sys
from collections import Counter
import numpy as np
import polars as pl

# 1 million random fruits
fruits_list = ["apple", "banana", "cherry"]
n = 1_000_000
np.random.seed(42)
random_fruits = np.random.choice(fruits_list, size=n)

# -------------------------
# Python list
# -------------------------
py_list = list(random_fruits)
start = time.time()
py_count = Counter(py_list)
py_time = time.time() - start
py_mem = sys.getsizeof(py_list) + sum(sys.getsizeof(x) for x in py_list)

# -------------------------
# NumPy array
# -------------------------
np_array = random_fruits
start = time.time()
np_count = np.unique(np_array, return_counts=True)
np_time = time.time() - start
np_mem = np_array.nbytes

# -------------------------
# Polars string column
# -------------------------
df_str = pl.DataFrame({"fruit_str": random_fruits})
start = time.time()
str_count = df_str.group_by("fruit_str").len()
str_time = time.time() - start
str_mem = df_str.estimated_size()

# -------------------------
# Polars categorical column
# -------------------------
df_cat = df_str.with_columns(pl.col("fruit_str").cast(pl.Categorical))
start = time.time()
cat_count = df_cat.group_by("fruit_str").len()
cat_time = time.time() - start
cat_mem = df_cat.estimated_size()

# -------------------------
# Results
# -------------------------
print(f"{'Type':<20} {'Memory (MB)':<15} {'Count Time (s)':<15}")
print(f"{'Python list':<20} {py_mem / 1e6:<15.2f} {py_time:<15.4f}")
print(f"{'NumPy array':<20} {np_mem / 1e6:<15.2f} {np_time:<15.4f}")
print(f"{'Polars string':<20} {str_mem / 1e6:<15.2f} {str_time:<15.4f}")
print(f"{'Polars categorical':<20} {cat_mem / 1e6:<15.2f} {cat_time:<15.4f}")

##Pivot Unpivot


In [ ]:
import polars as pl
from rich import print

🧠 Key Idea
Long format → TIDY, good for analysis

  Wide format (pivoted) → great for comparisons / dashboards

[Tidy Data](https://tidyr.tidyverse.org/articles/tidy-data.html) 🧹

1. Each variable forms a column.
2. Each observation forms a row.

---




In [ ]:
import polars as pl
from rich import print

# Sales data in long format
sales = pl.DataFrame({
    "date": ["2024-01-01", "2024-01-01", "2024-01-01",
             "2024-01-02", "2024-01-02", "2024-01-02",
             "2024-01-03", "2024-01-03", "2024-01-03"],
    "product": ["Widget", "Gadget", "Doohickey",
                "Widget", "Gadget", "Doohickey",
                "Widget", "Gadget", "Doohickey"],
    "region": ["East", "East", "West",
               "East", "West", "East",
               "West", "East", "West"],
    "quantity": [10, 15, 20, 12, 18, 22, 14, 16, 25],
    "revenue": [100, 225, 300, 120, 270, 330, 140, 240, 375]
})

print("Original Data (Long Format):")
print(sales)


Original Data (Long Format):

shape: (9, 5)
┌────────────┬───────────┬────────┬──────────┬─────────┐
│ date       ┆ product   ┆ region ┆ quantity ┆ revenue │
│ ---        ┆ ---       ┆ ---    ┆ ---      ┆ ---     │
│ str        ┆ str       ┆ str    ┆ i64      ┆ i64     │
╞════════════╪═══════════╪════════╪══════════╪═════════╡
│ 2024-01-01 ┆ Widget    ┆ East   ┆ 10       ┆ 100     │
│ 2024-01-01 ┆ Gadget    ┆ East   ┆ 15       ┆ 225     │
│ 2024-01-01 ┆ Doohickey ┆ West   ┆ 20       ┆ 300     │
│ 2024-01-02 ┆ Widget    ┆ East   ┆ 12       ┆ 120     │
│ 2024-01-02 ┆ Gadget    ┆ West   ┆ 18       ┆ 270     │
│ 2024-01-02 ┆ Doohickey ┆ East   ┆ 22       ┆ 330     │
│ 2024-01-03 ┆ Widget    ┆ West   ┆ 14       ┆ 140     │
│ 2024-01-03 ┆ Gadget    ┆ East   ┆ 16       ┆ 240     │
│ 2024-01-03 ┆ Doohickey ┆ West   ┆ 25       ┆ 375     │
└────────────┴───────────┴────────┴──────────┴─────────┘

````markdown
# 🔄 Polars Pivot Table — Basic Structure

---

## 📌 Pivot Syntax

```python
df.pivot(
    on="column_to_pivot",        # Values become new column headers
    index="row_identifier",      # Column(s) that define each row
    values="value_column",       # Values to aggregate into the pivot
    aggregate_function="sum"     # Aggregation applied to duplicates
)
````

---

## 🧠 Parameter Breakdown

* **`on`** → Column whose *unique values become new columns*
* **`index`** → Column(s) that define the *rows of the pivot table*
* **`values`** → Column containing the *data to aggregate*
* **`aggregate_function`** → How to combine multiple values in each cell

---

## ⚙️ Available Aggregate Functions

You can choose from the following built-in aggregation functions:

* `first` → takes the first value
* `last` → takes the last value
* `sum` → sums all values
* `min` → returns the minimum value
* `max` → returns the maximum value
* `mean` → computes the average
* `median` → computes the median
* `len` → counts the number of values

---

## 🚀 Key Insight

Pivoting in Polars transforms **long format data → wide format data**, while optionally aggregating duplicate entries using a specified function.

```
```


Sum Aggregation

In [ ]:
#sum revenue by product
pivot_sum = sales.pivot(
    on="product",
    index="date",
    values="revenue",
    aggregate_function="sum"
)

print("Pivot with Sum:")
print(pivot_sum)

Pivot with Sum:

shape: (3, 4)
┌────────────┬────────┬────────┬───────────┐
│ date       ┆ Widget ┆ Gadget ┆ Doohickey │
│ ---        ┆ ---    ┆ ---    ┆ ---       │
│ str        ┆ i64    ┆ i64    ┆ i64       │
╞════════════╪════════╪════════╪═══════════╡
│ 2024-01-01 ┆ 100    ┆ 225    ┆ 300       │
│ 2024-01-02 ┆ 120    ┆ 270    ┆ 330       │
│ 2024-01-03 ┆ 140    ┆ 240    ┆ 375       │
└────────────┴────────┴────────┴───────────┘

Mean Aggregation

In [ ]:
#average quantity by region & product
pivot_mean  = sales.pivot(
    on ="product",
    index="region",
    values="quantity",
    aggregate_function="mean"
)

print("Pivot with MEAN:")
print(pivot_mean)

Pivot with MEAN:

shape: (2, 4)
┌────────┬────────┬────────┬───────────┐
│ region ┆ Widget ┆ Gadget ┆ Doohickey │
│ ---    ┆ ---    ┆ ---    ┆ ---       │
│ str    ┆ f64    ┆ f64    ┆ f64       │
╞════════╪════════╪════════╪═══════════╡
│ East   ┆ 11.0   ┆ 15.5   ┆ 22.0      │
│ West   ┆ 14.0   ┆ 18.0   ┆ 22.5      │
└────────┴────────┴────────┴───────────┘

Max and Min Aggregation

In [ ]:
#min revenue by date and product
pivot_min = sales.pivot(
    on="product",
    index = "date",
    values="revenue",
    aggregate_function="min"
)

print("Pivot with MIN:")
print(pivot_min)

Pivot with MIN:

shape: (3, 4)
┌────────────┬────────┬────────┬───────────┐
│ date       ┆ Widget ┆ Gadget ┆ Doohickey │
│ ---        ┆ ---    ┆ ---    ┆ ---       │
│ str        ┆ i64    ┆ i64    ┆ i64       │
╞════════════╪════════╪════════╪═══════════╡
│ 2024-01-01 ┆ 100    ┆ 225    ┆ 300       │
│ 2024-01-02 ┆ 120    ┆ 270    ┆ 330       │
│ 2024-01-03 ┆ 140    ┆ 240    ┆ 375       │
└────────────┴────────┴────────┴───────────┘

In [ ]:
#max quantity by region and product
pivot_max = sales.pivot(
    on="product",
    index = "region",
    values="quantity",
    aggregate_function="max"
)

print("Pivot with MAX:")
print(pivot_max)

Pivot with MAX:

shape: (2, 4)
┌────────┬────────┬────────┬───────────┐
│ region ┆ Widget ┆ Gadget ┆ Doohickey │
│ ---    ┆ ---    ┆ ---    ┆ ---       │
│ str    ┆ i64    ┆ i64    ┆ i64       │
╞════════╪════════╪════════╪═══════════╡
│ East   ┆ 12     ┆ 16     ┆ 22        │
│ West   ┆ 14     ┆ 18     ┆ 25        │
└────────┴────────┴────────┴───────────┘

Median Aggregation

In [ ]:
#median revenue by product and region
pivot_median = sales.pivot(
    on="product",
    index="region",
    values="revenue",
    aggregate_function="median"
    )

print("Pivot with MEDIAN:")
print(pivot_median)

Pivot with MEDIAN:

shape: (2, 4)
┌────────┬────────┬────────┬───────────┐
│ region ┆ Widget ┆ Gadget ┆ Doohickey │
│ ---    ┆ ---    ┆ ---    ┆ ---       │
│ str    ┆ f64    ┆ f64    ┆ f64       │
╞════════╪════════╪════════╪═══════════╡
│ East   ┆ 110.0  ┆ 232.5  ┆ 330.0     │
│ West   ┆ 140.0  ┆ 270.0  ┆ 337.5     │
└────────┴────────┴────────┴───────────┘

Pivot on Multiple Indices

In [ ]:
multi_pivot = sales.pivot(
    on="product",
    index=["date", "region"],  # here we pass a list of columns
    values="revenue",
    aggregate_function="sum"
    )

print("Pivot With Multiple Indices:")
print(multi_pivot)

Pivot With Multiple Indices:

shape: (6, 5)
┌────────────┬────────┬────────┬────────┬───────────┐
│ date       ┆ region ┆ Widget ┆ Gadget ┆ Doohickey │
│ ---        ┆ ---    ┆ ---    ┆ ---    ┆ ---       │
│ str        ┆ str    ┆ i64    ┆ i64    ┆ i64       │
╞════════════╪════════╪════════╪════════╪═══════════╡
│ 2024-01-01 ┆ East   ┆ 100    ┆ 225    ┆ 0         │
│ 2024-01-01 ┆ West   ┆ 0      ┆ 0      ┆ 300       │
│ 2024-01-02 ┆ East   ┆ 120    ┆ 0      ┆ 330       │
│ 2024-01-02 ┆ West   ┆ 0      ┆ 270    ┆ 0         │
│ 2024-01-03 ┆ West   ┆ 140    ┆ 0      ┆ 375       │
│ 2024-01-03 ┆ East   ┆ 0      ┆ 240    ┆ 0         │
└────────────┴────────┴────────┴────────┴───────────┘

#Unpivot *(formerly melt)*

In [ ]:
unpivoted = pivot_sum.unpivot(
    on=["Doohickey", "Gadget", "Widget"],
    index="date",
    variable_name="product",
    value_name="revenue"
)
print("Unpivoted:")
print(unpivoted)

Unpivoted:

shape: (9, 3)
┌────────────┬───────────┬─────────┐
│ date       ┆ product   ┆ revenue │
│ ---        ┆ ---       ┆ ---     │
│ str        ┆ str       ┆ i64     │
╞════════════╪═══════════╪═════════╡
│ 2024-01-01 ┆ Doohickey ┆ 300     │
│ 2024-01-02 ┆ Doohickey ┆ 330     │
│ 2024-01-03 ┆ Doohickey ┆ 375     │
│ 2024-01-01 ┆ Gadget    ┆ 225     │
│ 2024-01-02 ┆ Gadget    ┆ 270     │
│ 2024-01-03 ┆ Gadget    ┆ 240     │
│ 2024-01-01 ┆ Widget    ┆ 100     │
│ 2024-01-02 ┆ Widget    ┆ 120     │
│ 2024-01-03 ┆ Widget    ┆ 140     │
└────────────┴───────────┴─────────┘

##  Sum_Horizontal

Because it is columnar and Rust based Polars is 2x - 10x faster than Pandas performing these operations. Here is a speed test result. I'll show you a cool demo in a minute.

In [ ]:
import polars as pl
import pandas as pd
pl.Config.set_tbl_cols(50)
from rich import print

golf =  {
    "player" : ["Arnie", "Jack", "Tiger"],
    "hole_1" : [ 3, 4, 4],
    "hole_2" : [ 4, 4, 4],
    "hole_3" : [ 5, 3, 4],
    "hole_4" : [ 3, 4, 3],
    "hole_5" : [ 3, 4, 4],
    "hole_6" : [ 3, 6, 4],
    "hole_7" : [ 4, 3, 4],
    "hole_8" : [ 5, 3, 4],
    "hole_9" : [ 3, 4, 4]
        }


In [ ]:
pddf = pd.DataFrame(golf)
pddf

,player,hole_1,hole_2,hole_3,hole_4,hole_5,hole_6,hole_7,hole_8,hole_9
0,Arnie,3,4,5,3,3,3,4,5,3
1,Jack,4,4,3,4,4,6,3,3,4
2,Tiger,4,4,4,3,4,4,4,4,4


Polars

In [ ]:
pldf  = pl.DataFrame(golf)
print(pldf)

shape: (3, 10)
┌────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┐
│ player ┆ hole_1 ┆ hole_2 ┆ hole_3 ┆ hole_4 ┆ hole_5 ┆ hole_6 ┆ hole_7 ┆ hole_8 ┆ hole_9 │
│ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    │
│ str    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    │
╞════════╪════════╪════════╪════════╪════════╪════════╪════════╪════════╪════════╪════════╡
│ Arnie  ┆ 3      ┆ 4      ┆ 5      ┆ 3      ┆ 3      ┆ 3      ┆ 4      ┆ 5      ┆ 3      │
│ Jack   ┆ 4      ┆ 4      ┆ 3      ┆ 4      ┆ 4      ┆ 6      ┆ 3      ┆ 3      ┆ 4      │
│ Tiger  ┆ 4      ┆ 4      ┆ 4      ┆ 3      ┆ 4      ┆ 4      ┆ 4      ┆ 4      ┆ 4      │
└────────┴────────┴────────┴────────┴────────┴────────┴────────┴────────┴────────┴────────┘

Row-wise sum across columns

pandas

In [ ]:
pddf['front_9_total'] =  pddf.filter(like="hole_").sum(axis=1)
pddf

,player,hole_1,hole_2,hole_3,hole_4,hole_5,hole_6,hole_7,hole_8,hole_9,front_9_total
0,Arnie,3,4,5,3,3,3,4,5,3,33
1,Jack,4,4,3,4,4,6,3,3,4,35
2,Tiger,4,4,4,3,4,4,4,4,4,35


polars

In [ ]:
# simple sum

front_9 = pldf.with_columns(
     pl.sum_horizontal(
        pl.col(pl.Int64)
    ).alias("front_9_total")
)

print(front_9)

shape: (3, 11)
┌────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┐
│ player ┆ hole_1 ┆ hole_2 ┆ hole_3 ┆ hole_4 ┆ hole_5 ┆ hole_6 ┆ hole_7 ┆ hole_8 ┆ hole_9 ┆ front_ │
│ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ 9_tota │
│ str    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ l      │
│        ┆        ┆        ┆        ┆        ┆        ┆        ┆        ┆        ┆        ┆ ---    │
│        ┆        ┆        ┆        ┆        ┆        ┆        ┆        ┆        ┆        ┆ i64    │
╞════════╪════════╪════════╪════════╪════════╪════════╪════════╪════════╪════════╪════════╪════════╡
│ Arnie  ┆ 3      ┆ 4      ┆ 5      ┆ 3      ┆ 3      ┆ 3      ┆ 4      ┆ 5      ┆ 3      ┆ 33     │
│ Jack   ┆ 4      ┆ 4      ┆ 3      ┆ 4      ┆ 4      ┆ 6      ┆ 3      ┆ 3      ┆ 4      ┆ 35     │
│ Tiger  ┆ 4      ┆ 4      ┆ 4      ┆ 3      ┆ 4      ┆ 4      ┆ 4      ┆ 4      ┆ 4      ┆ 35     │
└────────┴────────┴────────┴────────┴────────┴────────┴────────┴────────┴────────┴────────┴────────┘

In [ ]:
#other agg fuctions

result = pldf.with_columns(
    pl.mean_horizontal(pl.col(pl.Int64)).alias("avg_score"),
    pl.max_horizontal(pl.col(pl.Int64)).alias("worst_hole"),
    pl.min_horizontal(pl.col(pl.Int64)).alias("best_hole")
)
print(result)

shape: (3, 13)
┌─────┬─────┬─────┬────────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┐
│ pla ┆ hol ┆ hol ┆ hole_3 ┆ hole_ ┆ hole_ ┆ hole_ ┆ hole_ ┆ hole_ ┆ hole_ ┆ avg_s ┆ worst ┆ best_ │
│ yer ┆ e_1 ┆ e_2 ┆ ---    ┆ 4     ┆ 5     ┆ 6     ┆ 7     ┆ 8     ┆ 9     ┆ core  ┆ _hole ┆ hole  │
│ --- ┆ --- ┆ --- ┆ i64    ┆ ---   ┆ ---   ┆ ---   ┆ ---   ┆ ---   ┆ ---   ┆ ---   ┆ ---   ┆ ---   │
│ str ┆ i64 ┆ i64 ┆        ┆ i64   ┆ i64   ┆ i64   ┆ i64   ┆ i64   ┆ i64   ┆ f64   ┆ i64   ┆ i64   │
╞═════╪═════╪═════╪════════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╡
│ Arn ┆ 3   ┆ 4   ┆ 5      ┆ 3     ┆ 3     ┆ 3     ┆ 4     ┆ 5     ┆ 3     ┆ 3.666 ┆ 5     ┆ 3     │
│ ie  ┆     ┆     ┆        ┆       ┆       ┆       ┆       ┆       ┆       ┆ 667   ┆       ┆       │
│ Jac ┆ 4   ┆ 4   ┆ 3      ┆ 4     ┆ 4     ┆ 6     ┆ 3     ┆ 3     ┆ 4     ┆ 3.888 ┆ 6     ┆ 3     │
│ k   ┆     ┆     ┆        ┆       ┆       ┆       ┆       ┆       ┆       ┆ 889   ┆       ┆       │
│ Tig ┆ 4   ┆ 4   ┆ 4      ┆ 3     ┆ 4     ┆ 4     ┆ 4     ┆ 4     ┆ 4     ┆ 3.888 ┆ 4     ┆ 3     │
│ er  ┆     ┆     ┆        ┆       ┆       ┆       ┆       ┆       ┆       ┆ 889   ┆       ┆       │
└─────┴─────┴─────┴────────┴───────┴───────┴───────┴───────┴───────┴───────┴───────┴───────┴───────┘

#Speed Comparison 🐢 v ⚡


In [ ]:
import pandas as pd
import polars as pl
import numpy as np
import time


# create 1 million column 20 row df of random integers
rows = 1_000_000
cols = 20

data = {
    f"col_{i}": np.random.randint(1, 100, rows)
    for i in range(cols)
}

# pandas
pdf = pd.DataFrame(data)

# polars
pldf = pl.DataFrame(data)

In [ ]:
#pandas
start = time.perf_counter()

pdf["row_sum"] = pdf.sum(axis=1)

end = time.perf_counter()

print("Pandas:", end - start)

Pandas: 0.2578693949999433

In [ ]:
#polars
start = time.perf_counter()

result = pldf.with_columns(
    pl.sum_horizontal(pl.all()).alias("row_sum")
)

end = time.perf_counter()

print("Polars:", end - start)

Polars: 0.04793075999987195

# ⚡ Why Polars Wins vs Pandas (axis=1)

When doing **row-wise operations (horizontal computations)** like `sum(axis=1)`, Polars often significantly outperforms Pandas.

---

## 🐼 Pandas: Why It’s Slower

Pandas struggles with row-wise operations because:

### 🧠 Python-era row logic
- `axis=1` operations often fall back to Python-level loops
- Row-by-row processing reduces vectorization benefits

### 🧵 Less parallelized
- Limited multithreading for row-wise operations
- Many operations run in a single-threaded fashion

### 💾 Memory overhead
- Data is processed row-wise, increasing cache misses
- Intermediate object creation adds overhead

---

## 🦀 Polars: Why It’s Faster

Polars is built for performance from the ground up:

### ⚙️ Rust engine
- Core computation written in Rust (not Python)
- Near-zero overhead execution

### 📊 Vectorized horizontal kernels
- Row-wise operations are optimized internally
- Avoids Python loops entirely

### 🚀 Multithreaded by default
- Automatically distributes work across CPU cores
- Efficient use of modern hardware

---

## 📌 Summary

| Feature | Pandas | Polars |
|--------|--------|--------|
| Execution engine | Python | Rust |
| Row-wise ops | Slow (`axis=1`) | Fast (`sum_horizontal`) |
| Parallelism | Limited | Built-in multithreading |
| Memory efficiency | Moderate | High |
| Performance at scale | Degrades quickly | Scales well |

---

## 🧪 Takeaway

If you're doing heavy **row-wise analytics, feature engineering, or scoring systems**, Polars is usually the better choice.

Pandas is fine for small datasets, but Polars is built for speed at scale.

## What is a “fold” in Polars?

A **fold** is a way to **reduce multiple columns into a single value** using an **accumulator**.

You can think of it like this:

> **Start with an initial value, then combine columns one by one.**

In Polars, folds are **row-wise operations across columns**  
(not down rows like `sum()` or `mean()` usually are).

This makes folds especially useful for:
- Row-level calculations
- Combining many signals into one value
- Custom business logic that spans multiple columns

There really is no pandas equivalent.  We would have to do something like:



```
pddf["hole-1"] + pddf["hole_2"] + pddf["hole_3] ...




In [ ]:
pldf  = pl.DataFrame(golf)
print(pldf)

shape: (3, 10)
┌────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┐
│ player ┆ hole_1 ┆ hole_2 ┆ hole_3 ┆ hole_4 ┆ hole_5 ┆ hole_6 ┆ hole_7 ┆ hole_8 ┆ hole_9 │
│ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    │
│ str    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    │
╞════════╪════════╪════════╪════════╪════════╪════════╪════════╪════════╪════════╪════════╡
│ Arnie  ┆ 3      ┆ 4      ┆ 5      ┆ 3      ┆ 3      ┆ 3      ┆ 4      ┆ 5      ┆ 3      │
│ Jack   ┆ 4      ┆ 4      ┆ 3      ┆ 4      ┆ 4      ┆ 6      ┆ 3      ┆ 3      ┆ 4      │
│ Tiger  ┆ 4      ┆ 4      ┆ 4      ┆ 3      ┆ 4      ┆ 4      ┆ 4      ┆ 4      ┆ 4      │
└────────┴────────┴────────┴────────┴────────┴────────┴────────┴────────┴────────┴────────┘

In [ ]:
result = pldf.with_columns(
    pl.fold(
        acc=pl.lit(0),
        function = lambda acc, x: acc + x,
        exprs=pl.all().exclude("player"))
        .alias("out")
)
print(result)

shape: (3, 11)
┌────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬─────┐
│ player ┆ hole_1 ┆ hole_2 ┆ hole_3 ┆ hole_4 ┆ hole_5 ┆ hole_6 ┆ hole_7 ┆ hole_8 ┆ hole_9 ┆ out │
│ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ --- │
│ str    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i32 │
╞════════╪════════╪════════╪════════╪════════╪════════╪════════╪════════╪════════╪════════╪═════╡
│ Arnie  ┆ 3      ┆ 4      ┆ 5      ┆ 3      ┆ 3      ┆ 3      ┆ 4      ┆ 5      ┆ 3      ┆ 33  │
│ Jack   ┆ 4      ┆ 4      ┆ 3      ┆ 4      ┆ 4      ┆ 6      ┆ 3      ┆ 3      ┆ 4      ┆ 35  │
│ Tiger  ┆ 4      ┆ 4      ┆ 4      ┆ 3      ┆ 4      ┆ 4      ┆ 4      ┆ 4      ┆ 4      ┆ 35  │
└────────┴────────┴────────┴────────┴────────┴────────┴────────┴────────┴────────┴────────┴─────┘

Check if rows meet a condition

In [ ]:
#use | for any and & for all
result = pldf.with_columns(
    pl.fold(
        acc=pl.lit(False),
        function = lambda acc, x:  acc | (x > 5),
        exprs=pl.all().exclude("player")
    ).alias("gt 5")
)

print(result)

shape: (3, 11)
┌────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬───────┐
│ player ┆ hole_1 ┆ hole_2 ┆ hole_3 ┆ hole_4 ┆ hole_5 ┆ hole_6 ┆ hole_7 ┆ hole_8 ┆ hole_9 ┆ gt 5  │
│ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---   │
│ str    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ bool  │
╞════════╪════════╪════════╪════════╪════════╪════════╪════════╪════════╪════════╪════════╪═══════╡
│ Arnie  ┆ 3      ┆ 4      ┆ 5      ┆ 3      ┆ 3      ┆ 3      ┆ 4      ┆ 5      ┆ 3      ┆ false │
│ Jack   ┆ 4      ┆ 4      ┆ 3      ┆ 4      ┆ 4      ┆ 6      ┆ 3      ┆ 3      ┆ 4      ┆ true  │
│ Tiger  ┆ 4      ┆ 4      ┆ 4      ┆ 3      ┆ 4      ┆ 4      ┆ 4      ┆ 4      ┆ 4      ┆ false │
└────────┴────────┴────────┴────────┴────────┴────────┴────────┴────────┴────────┴────────┴───────┘

In [ ]:
#birdie = 3
result = pldf.with_columns(
    pl.fold(
        acc=pl.lit(0),
        function=lambda acc, x: acc + (x == 3).cast(pl.Int8),
        exprs=pl.col("^hole_.*$")
    ).alias("birdies")
)
print(result)

shape: (3, 11)
┌────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┐
│ player ┆ hole_1 ┆ hole_2 ┆ hole_3 ┆ hole_4 ┆ hole_5 ┆ hole_6 ┆ hole_7 ┆ hole_8 ┆ hole_9 ┆ birdie │
│ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ s      │
│ str    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ i64    ┆ ---    │
│        ┆        ┆        ┆        ┆        ┆        ┆        ┆        ┆        ┆        ┆ i32    │
╞════════╪════════╪════════╪════════╪════════╪════════╪════════╪════════╪════════╪════════╪════════╡
│ Arnie  ┆ 3      ┆ 4      ┆ 5      ┆ 3      ┆ 3      ┆ 3      ┆ 4      ┆ 5      ┆ 3      ┆ 5      │
│ Jack   ┆ 4      ┆ 4      ┆ 3      ┆ 4      ┆ 4      ┆ 6      ┆ 3      ┆ 3      ┆ 4      ┆ 3      │
│ Tiger  ┆ 4      ┆ 4      ┆ 4      ┆ 3      ┆ 4      ┆ 4      ┆ 4      ┆ 4      ┆ 4      ┆ 1      │
└────────┴────────┴────────┴────────┴────────┴────────┴────────┴────────┴────────┴────────┴────────┘